# ASL Testing Metrics Notebook

This notebook clones or reuses the repository, loads the shared held-out test split, restores each checkpoint, and computes the evaluation metrics requested for the first six model variants.

Metrics reported for every run:
- F1 per class
- Micro F1
- Macro F1
- Confusion matrix
- Accuracy
- Loss

Evaluated runs:
- Custom CNN with SGD + momentum
- Custom CNN with Adam
- InceptionV3 from scratch
- InceptionV3 transfer learning
- MobileNetV2 from scratch
- MobileNetV2 transfer learning

The 29-class MobileNetV2 fine-tuned checkpoint is intentionally left out of this main comparison and should be evaluated separately.

In [1]:
import json
import os
import subprocess
import sys
from collections import OrderedDict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

def discover_repo_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, Path('/kaggle/working')]
    for candidate in candidates:
        if (candidate / 'engine').exists() and (candidate / 'requirements.txt').exists():
            return candidate
    return cwd

REPO_URL = 'https://github.com/FrancOlano/ASL-Recognition-DL'
REPO_NAME = 'ASL-Recognition-DL'
REPO_ROOT = discover_repo_root()

if not (REPO_ROOT / 'engine').exists():
    target_root = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd().resolve()
    repo_dir = target_root / REPO_NAME
    if not repo_dir.exists():
        subprocess.check_call(['git', 'clone', REPO_URL, str(repo_dir)])
    REPO_ROOT = repo_dir

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from engine import config as cfg
from engine.dataset import get_data_loaders
from engine.model_factory import build_model

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
torch.manual_seed(cfg.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(cfg.SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Repository root: {REPO_ROOT}')
print(f'Device: {device}')

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
def resolve_dataset_root():
    kaggle_roots = [
        Path('/kaggle/input/datasets/grassknoted/asl-alphabet'),
        Path('/kaggle/input/asl-alphabet'),
        Path('/kaggle/input/asl_alphabet'),
        Path('/kaggle/input/grassknoted-asl-alphabet'),
        Path('/kaggle/input/grassknoted/asl-alphabet'),
    ]
    for root in kaggle_roots:
        if root.exists():
            train_root = root / 'asl_alphabet_train'
            if train_root.exists():
                return train_root
            return root

    local_root = REPO_ROOT / 'data' / 'processed'
    if local_root.exists():
        return local_root

    raise FileNotFoundError('Could not find a dataset root in Kaggle inputs or data/processed.')

dataset_root = resolve_dataset_root()
all_classes = sorted([path.name for path in dataset_root.iterdir() if path.is_dir()])
classes_to_keep = all_classes[:26] if len(all_classes) >= 26 else all_classes

cfg.KAGGLE = Path('/kaggle/working').exists()
cfg.PROJECT_ROOT = REPO_ROOT
cfg.DATA_DIR = dataset_root
cfg.CLASSES_TO_KEEP = classes_to_keep
cfg.MODEL_OUTPUT_DIR = REPO_ROOT / 'models' / 'checkpoints'
cfg.RESULTS_DIR = REPO_ROOT / 'results'

cfg.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(cfg.RESULTS_DIR / 'classes.json').write_text(json.dumps(classes_to_keep, indent=2))

print(f'Dataset root: {dataset_root}')
print(f'Classes used for evaluation: {len(classes_to_keep)}')
print(classes_to_keep)

In [ ]:
train_loader, val_loader, test_loader, train_dataset, val_dataset, test_dataset = get_data_loaders(
    data_dir=cfg.DATA_DIR,
    batch_size=cfg.BATCH_SIZE,
    num_workers=cfg.NUM_WORKERS,
    classes_to_keep=cfg.CLASSES_TO_KEEP,
)

class_names = train_loader.dataset.subset.dataset.classes
num_classes = len(class_names)
cfg.NUM_CLASSES = num_classes

print('Split sizes:')
print(f'  Train: {len(train_dataset)}')
print(f'  Validation: {len(val_dataset)}')
print(f'  Test: {len(test_dataset)}')
print(f'  Number of classes: {num_classes}')
print(f'  Class names: {class_names}')

In [ ]:
RESULTS_ROOT = REPO_ROOT / 'results' / 'testing_metrics'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
CONFUSION_DIR = RESULTS_ROOT / 'confusion_matrices'
CONFUSION_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_OVERRIDES = {
    'custom_cnn_sgd_momentum': None,
    'custom_cnn_adam': None,
    'inception_v3_scratch': None,
    'inception_v3_transfer': None,
    'mobilenet_v2_scratch': None,
    'mobilenet_v2_transfer': None,
}

MODEL_RUNS = OrderedDict([
    ('custom_cnn_sgd_momentum', {
        'label': 'Custom CNN with SGD + momentum',
        'architecture': 'custom_cnn',
        'pretrained': False,
        'checkpoint_candidates': [
            'best_custom_cnn_sgd_momentum.pth',
            'custom_cnn_sgd_momentum.pth',
            'cnn_sgd_momentum.pth',
            'best_custom_cnn_scratch.pth',
        ],
    }),
    ('custom_cnn_adam', {
        'label': 'Custom CNN with Adam',
        'architecture': 'custom_cnn',
        'pretrained': False,
        'checkpoint_candidates': [
            'best_custom_cnn_adam.pth',
            'custom_cnn_adam.pth',
            'cnn_adam.pth',
            'best_custom_cnn_scratch.pth',
        ],
    }),
    ('inception_v3_scratch', {
        'label': 'InceptionV3 from scratch',
        'architecture': 'inception_v3',
        'pretrained': False,
        'checkpoint_candidates': [
            'best_inception_v3_scratch.pth',
            'inception_v3_scratch.pth',
        ],
    }),
    ('inception_v3_transfer', {
        'label': 'InceptionV3 transfer learning',
        'architecture': 'inception_v3',
        'pretrained': True,
        'checkpoint_candidates': [
            'best_inception_v3_pretrained.pth',
            'inception_v3_pretrained.pth',
            'best_inception_v3_transfer.pth',
            'inception_v3_transfer.pth',
        ],
    }),
    ('mobilenet_v2_scratch', {
        'label': 'MobileNetV2 from scratch',
        'architecture': 'mobilenet_v2',
        'pretrained': False,
        'checkpoint_candidates': [
            'best_mobilenet_v2_scratch.pth',
            'mobilenet_v2_scratch.pth',
        ],
    }),
    ('mobilenet_v2_transfer', {
        'label': 'MobileNetV2 transfer learning',
        'architecture': 'mobilenet_v2',
        'pretrained': True,
        'checkpoint_candidates': [
            'best_mobilenet_v2_pretrained.pth',
            'mobilenet_v2_pretrained.pth',
            'best_mobilenet_v2_transfer.pth',
            'mobilenet_v2_transfer.pth',
        ],
    }),
])

def _extract_state_dict(raw_checkpoint):
    if isinstance(raw_checkpoint, dict):
        for key in ('state_dict', 'model_state_dict', 'model', 'net'):
            value = raw_checkpoint.get(key)
            if isinstance(value, dict):
                return value
    return raw_checkpoint

def _strip_module_prefix(state_dict):
    stripped = {}
    for key, value in state_dict.items():
        if key.startswith('module.'):
            stripped[key[len('module.'):]] = value
        else:
            stripped[key] = value
    return stripped

def find_checkpoint_path(run_key, candidate_names):
    override = CHECKPOINT_OVERRIDES.get(run_key)
    if override:
        override_path = Path(override)
        if override_path.is_file():
            return override_path

    search_roots = [
        REPO_ROOT / 'models' / 'checkpoints',
        REPO_ROOT / 'checkpoints',
        REPO_ROOT,
        Path('/kaggle/input'),
        Path('/kaggle/working'),
    ]

    direct_candidates = []
    for candidate_name in candidate_names:
        candidate_path = Path(candidate_name)
        if candidate_path.is_file():
            return candidate_path
        for root in search_roots:
            if root.exists():
                direct_path = root / candidate_name
                if direct_path.is_file():
                    return direct_path
                direct_candidates.append(direct_path)

    matches = []
    for root in search_roots:
        if not root.exists():
            continue
        for candidate_name in candidate_names:
            matches.extend(root.rglob(candidate_name))

    matches = sorted({path.resolve() for path in matches if path.is_file()})
    if matches:
        return matches[0]

    candidate_text = '\n'.join(f'- {candidate}' for candidate in candidate_names)
    searched_text = '\n'.join(f'- {path}' for path in direct_candidates[:20])
    raise FileNotFoundError(
        f'Could not resolve a checkpoint for {run_key}. Tried these candidate names:\n{candidate_text}\n\nSearched paths include:\n{searched_text}'
    )

def build_model_for_run(run_config):
    return build_model(
        model_type=run_config['architecture'],
        num_classes=num_classes,
        pretrained=run_config['pretrained'],
    ).to(device)

def load_model_from_checkpoint(run_key, run_config):
    checkpoint_path = find_checkpoint_path(run_key, run_config['checkpoint_candidates'])
    raw_checkpoint = torch.load(checkpoint_path, map_location=device)
    state_dict = _strip_module_prefix(_extract_state_dict(raw_checkpoint))
    model = build_model_for_run(run_config)
    model.load_state_dict(state_dict, strict=True)
    model.eval()
    return model, checkpoint_path

def confusion_matrix_from_labels(y_true, y_pred, class_count):
    matrix = np.zeros((class_count, class_count), dtype=np.int64)
    for true_label, predicted_label in zip(y_true, y_pred):
        matrix[int(true_label), int(predicted_label)] += 1
    return matrix

def compute_metrics_from_confusion_matrix(matrix):
    true_positive = np.diag(matrix).astype(np.float64)
    predicted_total = matrix.sum(axis=0).astype(np.float64)
    actual_total = matrix.sum(axis=1).astype(np.float64)

    precision = np.divide(
        true_positive,
        predicted_total,
        out=np.zeros_like(true_positive),
        where=predicted_total != 0,
    )
    recall = np.divide(
        true_positive,
        actual_total,
        out=np.zeros_like(true_positive),
        where=actual_total != 0,
    )
    f1_scores = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros_like(true_positive),
        where=(precision + recall) != 0,
    )

    total_true_positive = true_positive.sum()
    total_predictions = predicted_total.sum()
    total_actual = actual_total.sum()
    micro_precision = total_true_positive / total_predictions if total_predictions else 0.0
    micro_recall = total_true_positive / total_actual if total_actual else 0.0
    micro_f1 = (
        2 * micro_precision * micro_recall / (micro_precision + micro_recall)
        if (micro_precision + micro_recall) != 0
        else 0.0
    )

    return f1_scores, micro_f1, float(np.mean(f1_scores))

def save_confusion_heatmap(matrix, labels, title, output_path):
    fig, ax = plt.subplots(figsize=(14, 12))
    image = ax.imshow(matrix, interpolation='nearest', cmap='Blues')
    ax.set_title(title)
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    ticks = np.arange(len(labels))
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_yticklabels(labels)
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')
    ax.set_ylim(len(labels) - 0.5, -0.5)
    fig.tight_layout()
    fig.savefig(output_path, dpi=200, bbox_inches='tight')
    plt.close(fig)

print('Checkpoint resolver and metric helpers are ready.')

In [ ]:
criterion = torch.nn.CrossEntropyLoss()
evaluation_rows = []
per_class_rows = []
confusion_matrices = {}

for run_key, run_config in MODEL_RUNS.items():
    print('\n' + '=' * 90)
    print(run_config['label'])
    print('=' * 90)

    model, checkpoint_path = load_model_from_checkpoint(run_key, run_config)
    print(f'Loaded checkpoint: {checkpoint_path}')

    total_loss = 0.0
    total_samples = 0
    all_targets = []
    all_predictions = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            if isinstance(outputs, tuple):
                outputs = outputs[0]
            loss = criterion(outputs, labels)
            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size
            predictions = torch.argmax(outputs, dim=1)
            all_targets.append(labels.cpu().numpy())
            all_predictions.append(predictions.cpu().numpy())

    y_true = np.concatenate(all_targets)
    y_pred = np.concatenate(all_predictions)
    matrix = confusion_matrix_from_labels(y_true, y_pred, num_classes)
    per_class_f1, micro_f1, macro_f1 = compute_metrics_from_confusion_matrix(matrix)
    accuracy = float((y_true == y_pred).mean())
    average_loss = total_loss / total_samples if total_samples else 0.0

    confusion_matrices[run_key] = matrix.tolist()

    model_heatmap_path = CONFUSION_DIR / f'{run_key}_confusion_matrix.png'
    save_confusion_heatmap(matrix, class_names, run_config['label'], model_heatmap_path)

    evaluation_rows.append({
        'run_key': run_key,
        'model': run_config['label'],
        'architecture': run_config['architecture'],
        'checkpoint': str(checkpoint_path),
        'loss': average_loss,
        'accuracy': accuracy,
        'micro_f1': micro_f1,
        'macro_f1': macro_f1,
        'confusion_matrix_path': str(model_heatmap_path),
    })

    for class_name, f1_score in zip(class_names, per_class_f1):
        per_class_rows.append({
            'run_key': run_key,
            'model': run_config['label'],
            'class_name': class_name,
            'f1_score': float(f1_score),
        })

    print(f'Loss: {average_loss:.4f}')
    print(f'Accuracy: {accuracy:.4f}')
    print(f'Micro F1: {micro_f1:.4f}')
    print(f'Macro F1: {macro_f1:.4f}')
    print(f'Confusion matrix saved to: {model_heatmap_path}')

summary_df = pd.DataFrame(evaluation_rows).sort_values('model').reset_index(drop=True)
per_class_df = pd.DataFrame(per_class_rows)
per_class_pivot = per_class_df.pivot(index='class_name', columns='model', values='f1_score').reindex(class_names)

summary_csv = RESULTS_ROOT / 'summary_metrics.csv'
per_class_csv = RESULTS_ROOT / 'per_class_f1.csv'
confusions_json = RESULTS_ROOT / 'confusion_matrices.json'

summary_df.to_csv(summary_csv, index=False)
per_class_df.to_csv(per_class_csv, index=False)
confusions_json.write_text(json.dumps(confusion_matrices, indent=2))

print('\nSaved outputs:')
print(f'- {summary_csv}')
print(f'- {per_class_csv}')
print(f'- {confusions_json}')
print(f'- {CONFUSION_DIR}')

display(summary_df.round(4))

print('\nPer-class F1 table:')
display(per_class_pivot.round(4))

## 29-Class MobileNetV2 Evaluation

The fine-tuned MobileNetV2 checkpoint for the 29-class setup should be evaluated separately from the six-model comparison above.
If you want to add it here later, reuse the same helpers with `classes_to_keep = None`, load `best_mobilenet_v2_finetuned_29.pth`, and write its metrics to a separate results folder.